# ADE Extraction — QLoRA Fine-Tuning (Kaggle)

This notebook runs the full pipeline from `ade-extraction-lora.zip`: data prep, QLoRA training, and base-vs-fine-tuned evaluation.

**Before running:**
1. In this notebook's right sidebar, go to **Settings** → **Accelerator** → choose **GPU T4 x2** (or P100).
2. In **Settings** → **Internet**, turn it **ON** (needed to download the base model and dataset from Hugging Face).
3. Upload `ade-extraction-lora.zip` as a Kaggle Dataset: click **Add Input** (top right) → **Upload** → select the zip. It will appear under `/kaggle/input/<dataset-name>/`.
4. Run the cells below in order.


## 1. Locate and unzip the uploaded project

In [ ]:
import glob, shutil, os, zipfile

# Find the uploaded zip anywhere under /kaggle/input
zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
assert zip_candidates, 'No zip found under /kaggle/input — did you upload ade-extraction-lora.zip as a Dataset input?'
zip_path = zip_candidates[0]
print('Found:', zip_path)

work_dir = '/kaggle/working/ade-extraction-lora'
if os.path.exists(work_dir):
    shutil.rmtree(work_dir)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/kaggle/working/')

os.chdir(work_dir)
print('Working directory:', os.getcwd())
!ls -R .


## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt


## 3. (Optional) Weights & Biases
If you want experiment tracking, set your API key below. Otherwise, edit `configs/qlora_ade.yaml` and set `report_to: "none"` before training.

In [ ]:
# import wandb
# wandb.login(key='YOUR_KEY_HERE')


## 4. Verify the ADE Corpus V2 dataset schema
Run this before the full data-prep step — it confirms the config/field names the code expects actually match what's currently on the Hugging Face Hub (dataset schemas do occasionally change).

In [ ]:
from datasets import load_dataset

rel = load_dataset('ade_corpus_v2', 'Ade_corpus_v2_drug_ade_relation', split='train')
cls = load_dataset('ade_corpus_v2', 'Ade_corpus_v2_classification', split='train')

print('Relation config columns:', rel.column_names)
print('Relation example:', rel[0])
print()
print('Classification config columns:', cls.column_names)
print('Classification example:', cls[0])

# If column names or the label convention (0 = not related) differ from this output,
# update src/data/prepare_dataset.py accordingly before continuing.


## 5. Prepare the dataset

In [ ]:
!python src/data/prepare_dataset.py --config configs/qlora_ade.yaml


## 6. Train (QLoRA)
This is the long-running step. On a T4, expect somewhere in the range of 1–3 hours for the configured 12k training examples / 3 epochs — actual time depends on sequence lengths and GPU availability, so don't take this as a promise. Checkpoints save every 100 steps (see `configs/qlora_ade.yaml`), so if the Kaggle session times out, re-run this cell — `trainer.train(resume_from_checkpoint=True)` picks up from the latest checkpoint automatically if you set that in train.py, or point it at a specific checkpoint path.

In [ ]:
!python src/training/train.py --config configs/qlora_ade.yaml


## 7. Evaluate: base model vs fine-tuned
Loads both models back to back and runs the full test set through each. This will print a comparison table and write `results/comparison.json`.

In [ ]:
!python src/evaluation/evaluate.py \
    --base_model_id Qwen/Qwen2.5-3B-Instruct \
    --adapter_dir outputs/qwen25-3b-ade-lora \
    --test_file data/test.jsonl


## 8. Inspect results

In [ ]:
import json
with open('results/comparison.json') as f:
    results = json.load(f)

print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'predictions'} for k, v in results.items()}, indent=2))

# Look at a few individual predictions to sanity-check qualitatively, not just the aggregate numbers
for p in results['fine_tuned_model']['predictions'][:5]:
    print('---')
    print('Input:', p['input_sentence'])
    print('Gold:', p['gold'])
    print('Predicted:', p['predicted'])


## 9. Try it on your own sentence

In [ ]:
!python src/inference/inference.py \
    --sentence "The patient developed severe rhabdomyolysis after being started on high-dose atorvastatin."


## 10. Save your outputs before the session ends
Kaggle sessions are ephemeral — anything in `/kaggle/working` not explicitly saved as Notebook Output or pushed elsewhere disappears. Recommended:
- Commit the notebook (Kaggle keeps `/kaggle/working` contents from the last commit as "Output").
- Push the LoRA adapter (`outputs/qwen25-3b-ade-lora`) to the Hugging Face Hub so it's not stuck in this session.
- Copy `results/comparison.json` out (e.g. download it, or commit the notebook) so you have your real numbers for the README/resume.

In [ ]:
# Optional: push the adapter to the Hugging Face Hub
# from huggingface_hub import login
# login(token='YOUR_HF_TOKEN')
# from peft import PeftModel
# Uncomment and adjust once you're ready:
# model.push_to_hub('your-username/qwen25-3b-ade-lora')
# tokenizer.push_to_hub('your-username/qwen25-3b-ade-lora')
